In [ ]:
# === Notebook common preamble (load the llm_math package) ===
import sys
from pathlib import Path

# Candidate paths for the llm_math package
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# Add parent directories as candidates (when running from the notebooks folder)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Try importing llm_math
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[Note] failed to load the llm_math package: {e}")
    print("  Clone the GitHub repository and run colab_setup.sh.")
# === end preamble ===


# Ch 22. DPO and Modern Alignment Methods — Aligning without RLHF

> **Learning Goals**
> - Derive how DPO reduces the three-stage RLHF pipeline to one loss
> - Understand the relationship between the Bradley-Terry model and the DPO loss
> - Compare modern alignment methods such as KTO, IPO, and ORPO

## 22.1 RLHF Complexity and the Rise of DPO

Problems with RLHF:
- Four models (policy, reference, RM, value), causing memory blow-up
- PPO is hyperparameter-sensitive and unstable
- Training a separate RM accumulates errors

**DPO** (Direct Preference Optimization, Rafailov et al., 2023):
- Combines RM training and PPO into a **single loss**
- Trains directly from preference data
- Requires only two models in memory: policy and reference


In [ ]:

# Shared model definition, same as Ch 18
import torch
import torch.nn as nn
import torch.nn.functional as F

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model)
        )
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def attention(self, x, mask):
        B, T, D = x.shape
        q, k, v = self.W_qkv(x).chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        scores = q @ k.transpose(-1, -2) / (self.d_k ** 0.5) + mask
        weights = F.softmax(scores, dim=-1)
        out = (weights @ v).transpose(1, 2).contiguous().view(B, T, D)
        return self.W_o(out)

    def forward(self, x, mask):
        x = x + self.dropout(self.attention(self.ln1(x), mask))
        x = x + self.dropout(self.ffn(self.ln2(x)))
        return x

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_heads=4, n_layers=2,
                 d_ff=256, max_seq_len=128, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight  # weight tying
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T = x.shape
        positions = torch.arange(T, device=x.device).unsqueeze(0)
        emb = self.token_emb(x) * (self.d_model ** 0.5) + self.pos_emb(positions)
        h = self.dropout(emb)
        mask = torch.triu(torch.full((T, T), float('-inf'), device=x.device), diagonal=1)
        for block in self.blocks:
            h = block(h, mask)
        h = self.ln_f(h)
        return self.lm_head(h)

print("MiniGPT Model definition complete")


In [ ]:
# vocabulary size setting (Ch 22 whole)
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

vocab_size = 100  # toy
print("vocab_size (toy):", vocab_size)


## 22.2 Review: Bradley-Terry Reward Model

$$P(\mathbf{y}_w \succ \mathbf{y}_l | \mathbf{x}) = \sigma(r(\mathbf{x}, \mathbf{y}_w) - r(\mathbf{x}, \mathbf{y}_l))$$

RM loss: $\mathcal{L}_{\text{RM}} = -\log \sigma(r_w - r_l)$

## 22.3 Deriving the Reward-Policy Relationship

KL-constrained optimization:
$$\max_\pi \mathbb{E}_{\mathbf{y} \sim \pi}[r(\mathbf{x}, \mathbf{y})] - \beta D_{\text{KL}}(\pi(\cdot|\mathbf{x}) \| \pi_{\text{ref}}(\cdot|\mathbf{x}))$$

The solution is:
$$\pi^*(\mathbf{y}|\mathbf{x}) = \frac{\pi_{\text{ref}}(\mathbf{y}|\mathbf{x}) \exp(r(\mathbf{x}, \mathbf{y}) / \beta)}{Z(\mathbf{x})}$$

Rearranging:
$$r(\mathbf{x}, \mathbf{y}) = \beta \log \frac{\pi^*(\mathbf{y}|\mathbf{x})}{\pi_{\text{ref}}(\mathbf{y}|\mathbf{x})} + \beta \log Z(\mathbf{x})$$

$\log Z(\mathbf{x})$ does not depend on $\mathbf{y}$, so it cancels when comparing $\mathbf{y}_w$ and $\mathbf{y}_l$.

## 22.4 DPO Loss

Substitute the reward-policy relationship into Bradley-Terry:

$$P(\mathbf{y}_w \succ \mathbf{y}_l | \mathbf{x}) = \sigma\left(\beta \log \frac{\pi_\theta(\mathbf{y}_w|\mathbf{x})}{\pi_{\text{ref}}(\mathbf{y}_w|\mathbf{x})} - \beta \log \frac{\pi_\theta(\mathbf{y}_l|\mathbf{x})}{\pi_{\text{ref}}(\mathbf{y}_l|\mathbf{x})}\right)$$

**DPO loss**:
$$\mathcal{L}_{\text{DPO}} = -\mathbb{E}\left[\log \sigma\left(\beta \log \frac{\pi_\theta(\mathbf{y}_w|\mathbf{x})}{\pi_{\text{ref}}(\mathbf{y}_w|\mathbf{x})} - \beta \log \frac{\pi_\theta(\mathbf{y}_l|\mathbf{x})}{\pi_{\text{ref}}(\mathbf{y}_l|\mathbf{x})}\right)\right]$$

Key point: train $\pi_\theta$ directly **without an RM**, using only log-probability ratios.


In [ ]:
# DPO loss implementation
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

def get_seq_logprob(model, x, y_mask=None):
    """sum of sequence log probabilities."""
    logits = model(x)
    log_probs = F.log_softmax(logits, dim=-1)
    # shift
    log_probs = log_probs[:, :-1, :]
    targets = x[:, 1:]
    # gather
    seq_log_probs = log_probs.gather(2, targets.unsqueeze(-1)).squeeze(-1)  # (B, T-1)
    if y_mask is not None:
        seq_log_probs = seq_log_probs * y_mask[:, 1:]
    return seq_log_probs.sum(dim=-1)  # (B,)

def dpo_loss(policy_chosen_logp, policy_rejected_logp,
             ref_chosen_logp, ref_rejected_logp, beta=0.1):
    """DPO Loss."""
    chosen_ratio = policy_chosen_logp - ref_chosen_logp
    rejected_ratio = policy_rejected_logp - ref_rejected_logp
    logits = beta * (chosen_ratio - rejected_ratio)
    return -F.logsigmoid(logits).mean()

# simulation
# synthetic log probability (computed by the model in practice)
torch.manual_seed(0)
B = 8
ref_chosen_logp = torch.randn(B) - 5  # chosen log probability under the reference model
ref_rejected_logp = torch.randn(B) - 7  # rejected usually lower

# initial policy: similar to the reference
policy_chosen_logp = ref_chosen_logp.clone() + torch.randn(B) * 0.1
policy_rejected_logp = ref_rejected_logp.clone() + torch.randn(B) * 0.1

# Training simulation (policy toward chosen responses)
losses = []
for step in range(100):
    # policy gradually toward chosen responses (gradient simulation)
    policy_chosen_logp = policy_chosen_logp + 0.05
    # gradient ascent on chosen ratio

    loss = dpo_loss(policy_chosen_logp, policy_rejected_logp,
                    ref_chosen_logp, ref_rejected_logp, beta=0.1)
    losses.append(loss.item())

import matplotlib.pyplot as plt
plt.figure(figsize=(9, 3))
plt.plot(losses)
plt.xlabel('Step'); plt.ylabel('DPO Loss')
plt.title('DPO Training simulation')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch22_dpo_loss.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"initial loss: {losses[0]:.4f}, final loss: {losses[-1]:.4f}")
print(f"Chosen/rejected log-prob difference: {(policy_chosen_logp - policy_rejected_logp).mean():.4f}")


## 22.5 DPO Training Loop

Core steps:
1. Freeze the SFT model as $\pi_{\text{ref}}$
2. Batch preference data $(\mathbf{x}, \mathbf{y}_w, \mathbf{y}_l)$
3. Compute log probabilities for $\mathbf{y}_w$ and $\mathbf{y}_l$ under both $\pi_\theta$ and $\pi_{\text{ref}}$
4. Compute DPO loss and backpropagate only through $\pi_\theta$


In [ ]:
# DPO training (simplified)
import torch
import torch.nn as nn
import torch.nn.functional as F

# synthetic preference data generation
def make_pref_batch(vocab_size, batch_size=4, seq_len=32):
    """generate chosen and rejected sequences."""
    chosen = torch.randint(0, vocab_size, (batch_size, seq_len))
    # rejected chosen in change some tokens
    rejected = chosen.clone()
    n_change = seq_len // 4
    for i in range(batch_size):
        idxs = torch.randperm(seq_len)[:n_change]
        rejected[i, idxs] = torch.randint(0, vocab_size, (n_change,))
    return chosen, rejected

# Model 2 (policy, reference)
torch.manual_seed(0)
policy = MiniGPT(vocab_size, d_model=32, n_heads=4, n_layers=2, d_ff=128, max_seq_len=64)
ref = MiniGPT(vocab_size, d_model=32, n_heads=4, n_layers=2, d_ff=128, max_seq_len=64)
ref.load_state_dict(policy.state_dict())  # same initially
for p in ref.parameters():
    p.requires_grad = False  # ref frozen

opt = torch.optim.AdamW(policy.parameters(), lr=5e-4, weight_decay=0.01)

losses = []
for step in range(50):
    chosen, rejected = make_pref_batch(vocab_size, batch_size=4, seq_len=32)
    # log probability over the whole sequence
    with torch.no_grad():
        ref_chosen_logp = get_seq_logprob(ref, chosen)
        ref_rejected_logp = get_seq_logprob(ref, rejected)
    policy_chosen_logp = get_seq_logprob(policy, chosen)
    policy_rejected_logp = get_seq_logprob(policy, rejected)

    loss = dpo_loss(policy_chosen_logp, policy_rejected_logp,
                    ref_chosen_logp, ref_rejected_logp, beta=0.1)
    opt.zero_grad()
    loss.backward()
    opt.step()
    losses.append(loss.item())

plt.figure(figsize=(9, 3))
plt.plot(losses)
plt.xlabel('Step'); plt.ylabel('DPO Loss')
plt.title('DPO Training (MiniGPT)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch22_dpo_training.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Final DPO loss: {losses[-1]:.4f}")


## 22.6 IPO, KTO, ORPO — DPO Variants

### IPO (Identity Preference Optimization)
Addresses DPO overfitting:
$$\mathcal{L}_{\text{IPO}} = (\log \sigma(\ldots) - \frac{1}{2})^2$$

### KTO (Kahneman-Tversky Optimization)
Does **not require preference pairs**. It only needs good/bad labels for individual responses:
$$\mathcal{L}_{\text{KTO}} = \sigma(\beta \log \frac{\pi}{\pi_{\text{ref}}} - z) + \lambda \sigma(z - \beta \log \frac{\pi}{\pi_{\text{ref}}})$$

This makes data collection easier because pair matching is unnecessary.

### ORPO (Odds Ratio Preference Optimization)
Combines SFT and DPO, and does not even require a reference model. It is highly efficient.


## 22.7 DPO vs PPO Comparison

| Item | PPO (RLHF) | DPO |
|---|---|---|
| Number of models | 4 (policy, ref, RM, value) | 2 (policy, ref) |
| Training stability | unstable | stable |
| Hyperparameters | many | few (mostly β) |
| Memory | high | low |
| Implementation complexity | high | low |
| Quality | can be slightly better | similar |

In practice, DPO is increasingly standard. Modern models such as Mistral and LLaMA-3 also use DPO-style alignment.


## 22.8 Key Takeaways

| Method | Core Idea | Data |
|---|---|---|
| RLHF | RM + PPO + KL | preference pairs |
| DPO | single loss from reward-policy relation | preference pairs |
| IPO | DPO + regularization | preference pairs |
| KTO | only good/bad labels | single labels |
| ORPO | combines SFT and DPO | preference pairs |

## Exercises

1. Compare DPO loss with $\beta = 0.01, 0.1, 1.0$ and explain the effect.
2. Explain how the Bradley-Terry model represents preference pairs.
3. Compare DPO and PPO memory usage.
4. Explain why KTO can train without preference pairs.
5. Derive the reward-policy relationship $r = \beta \log \frac{\pi^*}{\pi_{\text{ref}}} + \beta \log Z$.

> Solutions: `solutions/ch22_solutions.ipynb`
